In [3]:
import pandas as pd

movies_df = pd.read_csv("../dataset/movies.csv")
movies_df.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [10]:
ratings_df = pd.read_csv("../dataset/ratings.csv")
print(ratings_df.head())
ratings_sampled = ratings_df.sample(n=100000, random_state=0)

ratings_sampled.head()

   userId  movieId  rating   timestamp
0       1      296     5.0  1147880044
1       1      306     3.5  1147868817
2       1      307     5.0  1147868828
3       1      665     5.0  1147878820
4       1      899     3.5  1147868510


,userId,movieId,rating,timestamp
76998,594,519,3.0,836177816
13988377,90666,87430,2.0,1425221038
645617,4421,5502,3.0,1416152606
11081841,72074,2321,5.0,938652868
7789850,50602,7458,1.0,1419369317


In [ ]:
# df = ratings_sampled_df.merge(movies_df, on="movieId", how="left")
df = ratings_sampled.pivot_table(index="userId", columns="movieId", values="rating")

df
# from scipy.sparse import coo_matrix
# from scipy.sparse.linalg import svds
# from sklearn.metrics.pairwise import cosine_similarity
# import numpy as np


# mu = ratings_sampled["rating"].mean()

# item_bias = ratings_sampled.groupby("movieId")["rating"].mean() - mu

# temp = ratings_sampled.merge(item_bias.rename("b_i"), on="movieId", how="left")


# # calculate r_ui - mu - b_i for every rating
# temp["residual"] = temp["rating"] - mu - temp["b_i"]


# user_bias = temp.groupby("userId")["residual"].mean()


# # Step 1 — build sparse matrix
# users = ratings_sampled["userId"].astype("category")
# movies = ratings_sampled["movieId"].astype("category")

# # print(users)

# # sorted unique list of users and movies
# user_cats = users.cat.categories
# movie_cats = movies.cat.categories

# # print(user_cats)

# n_users = len(user_cats)
# n_items = len(movie_cats)


# # print(user_bias)
# # print(user_cats)

# # reorder for safety
# b_u = user_bias.reindex(user_cats).fillna(0).to_numpy()
# b_i = item_bias.reindex(movie_cats).fillna(0).to_numpy()


# # indices for each rating
# row = users.cat.codes.to_numpy()
# col = movies.cat.codes.to_numpy()

# # # Step 3 — residuals matrix: r_ui - mu - b_u - b_i
# ratings_arr = ratings_sampled["rating"].to_numpy()
# residuals = ratings_arr - (mu + b_u[row] + b_i[col])

# R_tilde = coo_matrix((residuals, (row, col)), shape=(n_users, n_items))

# # Step 4 — SVD on residuals
# U, s, Vt = svds(R_tilde, k=50)

# S_sqrt = np.sqrt(s)
# P = U * S_sqrt[np.newaxis, :]  # user latent factors
# Q = Vt.T * S_sqrt[np.newaxis, :]  # item latent factors

# # mappings: id -> index
# userid_to_idx = dict(zip(user_cats, range(n_users)))
# itemid_to_idx = dict(zip(movie_cats, range(n_items)))

# userid_to_idx

movieId,1,2,3,4,5,6,7,8,9,10,...,206089,206097,206210,207053,207057,207071,207830,207912,208689,208737
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
162534,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162536,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
162537,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np


def compute_latent_factors(ratings_mat, k=50):

    mu = ratings_mat["rating"].mean()

    item_bias = ratings_mat.groupby("movieId")["rating"].mean() - mu

    temp = ratings_mat.merge(item_bias.rename("b_i"), on="movieId", how="left")

    # calculate r_ui - mu - b_i for every rating
    temp["residual"] = temp["rating"] - mu - temp["b_i"]

    user_bias = temp.groupby("userId")["residual"].mean()

    users = ratings_mat["userId"].astype("category")
    movies = ratings_mat["movieId"].astype("category")

    # sorted unique list of users and movies
    user_cats = users.cat.categories
    movie_cats = movies.cat.categories

    n_users = len(user_cats)
    n_items = len(movie_cats)

    b_u = user_bias.reindex(user_cats).fillna(0).to_numpy()
    b_i = item_bias.reindex(movie_cats).fillna(0).to_numpy()

    # indices for each rating
    row = users.cat.codes.to_numpy()
    col = movies.cat.codes.to_numpy()

    # residuals matrix: r_ui - mu - b_u - b_i
    ratings_arr = ratings_mat["rating"].to_numpy()
    residuals = ratings_arr - (mu + b_u[row] + b_i[col])

    R_tilde = coo_matrix((residuals, (row, col)), shape=(n_users, n_items))
    print(R_tilde)

    # SVD on residuals
    U, s, Vt = svds(R_tilde, k=k)

    S_sqrt = np.sqrt(s)
    P = U * S_sqrt[np.newaxis, :]  # user latent factors
    Q = Vt.T * S_sqrt[np.newaxis, :]  # item latent factors

    print(P)
    print(Q)

    # mappings: id -> index
    userid_to_idx = dict(zip(user_cats, range(n_users)))
    itemid_to_idx = dict(zip(movie_cats, range(n_items)))

    return mu, b_u, b_i, P, Q, userid_to_idx, itemid_to_idx

In [22]:
ratings_sampled.head()

,userId,movieId,rating,timestamp
76998,594,519,3.0,836177816
13988377,90666,87430,2.0,1425221038
645617,4421,5502,3.0,1416152606
11081841,72074,2321,5.0,938652868
7789850,50602,7458,1.0,1419369317


In [14]:
mu, b_u, b_i, P, Q, userid_to_idx, itemid_to_idx = compute_latent_factors(
    ratings_sampled, k=50
)

<COOrdinate sparse matrix of dtype 'float64'
	with 100000 stored elements and shape (55079, 10181)>
  Coords	Values
  (178, 457)	0.0
  (30731, 7660)	-0.5036630036630036
  (1443, 3979)	-0.39898531595446274
  (24518, 1779)	0.0
  (17195, 4978)	0.0
  (36087, 1531)	0.0
  (8551, 219)	-0.1956313635004565
  (7189, 8532)	-0.25
  (3425, 1835)	-0.1775108225108224
  (1676, 5689)	-0.4084848484848487
  (24907, 7369)	1.353991596638655
  (5158, 3099)	0.0
  (28350, 2376)	-0.8575928584399968
  (41638, 7735)	0.0
  (31228, 1320)	0.0
  (9895, 6322)	-0.5177927927927932
  (52508, 274)	0.0
  (45314, 6892)	-0.5748728617660657
  (17013, 7225)	0.26579596239790426
  (10755, 942)	0.3630385487528347
  (45578, 4368)	0.29241183652948344
  (53382, 5819)	-1.3444919366347938
  (23610, 146)	-0.01843683839096677
  (8582, 7397)	-0.10023273035015379
  (33399, 4829)	0.7007575757575761
  :	:
  (42849, 4680)	0.0
  (27677, 7059)	0.0
  (37753, 5119)	0.13536210674061167
  (34624, 318)	0.0
  (31271, 936)	-0.8150485369081002
  (203

In [ ]:
def svd_recommender(user_id, n=10):
    if user_id not in userid_to_idx:
        raise ValueError(f"user_id {user_id} not in training data")

    u_idx = userid_to_idx[user_id]

    # all movie indices & ids
    n_items = Q.shape[0]
    movie_ids = [None] * n_items

    for movie_id, idx in itemid_to_idx.items():
        movie_ids[idx] = movie_id

    movie_ids = np.array(movie_ids)

    # movies the user has already rated
    rated_movies = set(
        ratings_sampled.loc[ratings_sampled["userId"] == user_id, "movieId"].unique()
    )

    # compute predicted ratings for ALL items for this user
    pu = P[u_idx]
    bu = b_u[u_idx]

    # vectorised: mu + b_u + b_i + p_u^T q_i
    scores = (mu + bu + b_i + Q @ pu).:2f

    preds = pd.Series(scores, index=movie_ids)


    # filter out already-seen items
    preds_unseen = preds[~preds.index.isin(rated_movies)]

    # top-n
    top_preds = preds_unseen.sort_values(ascending=False).head(n)


    recs = top_preds.reset_index()
    recs.columns = ["movieId", "predicted_rating"]

    # attach titles
    recs = recs.merge(movies_df[["movieId", "title"]], on="movieId", how="left")
    recs = recs[["movieId", "title", "predicted_rating"]]

    return recs

In [ ]:
user_id = 594
n = 10
recs = svd_recommender(user_id, n)
recs

,movieId,title,predicted_rating
0,203519,Fast & Furious Presents: Hobbs & Shaw (2019),5.583333
1,7983,Broadway Danny Rose (1984),5.583333
2,8011,"Weather Underground, The (2002)",5.583333
3,157375,The Cutting Edge: Fire & Ice (2010),5.583333
4,157146,Dial H-I-S-T-O-R-Y (1997),5.583333
5,1134,Johnny 100 Pesos (Johnny cien pesos) (1993),5.583333
6,1312,Female Perversions (1996),5.583333
7,1311,Santa with Muscles (1996),5.583333
8,1317,I'm Not Rappaport (1996),5.583333
9,152617,Project-M (2014),5.583333
